# FM回归
将AED与fm_reg_controls表进合并，用于进行回归  

## 导入库

In [91]:
import polars as pl
import dotenv
import os
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from statsmodels.tools.tools import add_constant
import warnings
dotenv.load_dotenv()

True

## 超参数

In [92]:
TASK_PREFIX = 'mech1'
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_PREFIX}'
SAVE = True
HETER = True # 如果不使用异质性，则导入MA_copy，保证不存在异质性

# 缩尾比例
WINSORIZE_RATIO = 0.01

DATABASE_URL = os.getenv('POSTGRES_URL')

## 读取数据
- 从数据库中读取控制变量  
- 从BASE_LINE_REG_DIR中获取aed.parquet 

### 读数据库 

In [93]:
# 使用polars从数据库读取fm_reg_controls表数据，指定schema避免类型推断问题
fm_reg_controls_schema = {
    'stkcd': pl.Utf8,              # 股票代码
    'accper': pl.Date,             # 会计期间
    'betavals': pl.Float64,        # 贝塔值
    'bm_ratio': pl.Float64,        # 账面市值比
    'gross_margin': pl.Float64,    # 毛利率
    'investment_ratio': pl.Float64, # 投资比率
    'ln_market_value': pl.Float64   # 对数市值
}

fm_reg_controls_df = pl.read_database_uri(
    query="SELECT * FROM statics.fm_reg_controls",
    uri=DATABASE_URL,
    schema_overrides=fm_reg_controls_schema
)

fm_reg_controls_df.head()

stkcd,accper,betavals,bm_ratio,gross_margin,investment_ratio,ln_market_value
str,date,f64,f64,f64,f64,f64
"""000001""",1997-01-01,0.93259,null,null,null,16.435539
"""000001""",1997-02-01,0.88586,null,null,null,16.433457
"""000001""",1997-03-01,1.06864,null,null,null,16.785434
"""000001""",1997-04-01,1.01343,null,null,null,17.082685
"""000001""",1997-05-01,1.22579,null,null,null,16.994263


### 读取AED因子

In [94]:
ma_df = pl.read_parquet(SAVE_BASE_DIR + '/MA因子.parquet') if HETER else pl.read_parquet(SAVE_BASE_DIR + '/MA因子_copy.parquet')
ma_df.head()

date,portfolio,MA,return
date,str,f64,f64
2022-12-01,"""300148""",0.52728,-0.0612
2024-02-01,"""301188""",0.617975,0.014964
2020-01-01,"""600056""",0.432743,0.0076
2016-03-01,"""002126""",0.668147,0.080424
2020-10-01,"""002091""",0.859207,0.0894


## 处理数据 
1.截面标准化
 

In [95]:
# 标准化函数
def cs_zscore_polars(
    df: pl.DataFrame,
    date_col: str,
    code_col: str,
    x_cols: list[str],
    *,
    ddof: int = 0,
    eps: float = 1e-12,
    keep_other_cols: bool = True,
) -> pl.DataFrame:
    """
    按 date_col 分组，对 x_cols 做截面标准化（z-score）。

    - ddof: 标准差自由度（0 对应总体std，1 对应样本std）
    - eps: 防止 std=0 或 null 导致除零
    - keep_other_cols: 是否保留除 date/code/x_cols 之外的其他列
    """
    # 需要保留的列
    base_cols = [date_col, code_col]
    if keep_other_cols:
        keep_cols = df.columns
    else:
        keep_cols = list(dict.fromkeys(base_cols + x_cols))

    # 预聚合：每个 date 的均值 & 标准差
    agg_exprs = []
    for c in x_cols:
        agg_exprs.append(pl.col(c).mean().alias(f"__mean__{c}"))
        agg_exprs.append(pl.col(c).std(ddof=ddof).alias(f"__std__{c}"))

    stats = df.select([date_col, *x_cols]).group_by(date_col).agg(agg_exprs)

    # join 回原表，再计算 z-score
    out = (
        df.select(keep_cols)
        .join(stats, on=date_col, how="left")
        .with_columns(
            [
                (
                    (pl.col(c) - pl.col(f"__mean__{c}"))
                    / pl.when(pl.col(f"__std__{c}").is_null() | (pl.col(f"__std__{c}") <= 0))
                    .then(pl.lit(eps))
                    .otherwise(pl.col(f"__std__{c}"))
                ).alias(c)
                for c in x_cols
            ]
        )
        .drop([f"__mean__{c}" for c in x_cols] + [f"__std__{c}" for c in x_cols])
    )

    return out

ma_df = cs_zscore_polars(ma_df, 'date', 'portfolio', ['MA'])
fm_reg_controls_df = cs_zscore_polars(fm_reg_controls_df, 'accper', 'stkcd', ['betavals','bm_ratio','gross_margin','investment_ratio','ln_market_value'])

2.对于Controls数据，由于部分数据是季频，用该季数据填充该月数据 

In [96]:
### 季度数据（使用滞后可得信息，但不补全：仅在季末月使用上一季度末值）

# 季频字段：只在 3/6/9/12 月出现；例如 1997-06 使用 1997-03 的值
quarterly_columns = ['bm_ratio', 'gross_margin', 'investment_ratio']

base_controls = (
    fm_reg_controls_df
    .with_columns(pl.col('accper').cast(pl.Date))
    .rename({'accper': 'date', 'stkcd': 'portfolio'})
)

# 取季末月的季频观测
q_end = (
    base_controls
    .with_columns(pl.col('date').dt.month().alias('_m'))
    .filter(pl.col('_m').is_in([3, 6, 9, 12]))
    .select(['portfolio', 'date', *quarterly_columns])
)

# 把“上一季度末值”对齐到“下一季度末月”（不生成季度内其他月份）
# 3->6, 6->9, 9->12, 12->next year 3
lagged_quarterly = (
    q_end
    .with_columns(pl.col('date').dt.offset_by('3mo').alias('date'))
    .sort(['portfolio', 'date'])
    .unique(subset=['portfolio', 'date'])
)

# 回填到月度控制变量表：非季末月的季度变量保持为 null
fm_reg_controls_monthly_df = (
    base_controls
    .drop(quarterly_columns)
    .join(lagged_quarterly, on=['portfolio', 'date'], how='left')
)

fm_reg_controls_monthly_df.head(10)

portfolio,date,betavals,ln_market_value,bm_ratio,gross_margin,investment_ratio
str,date,f64,f64,f64,f64,f64
"""000001""",1997-01-01,0.038588,4.757128,null,null,null
"""000001""",1997-02-01,-0.022514,4.753201,null,null,null
"""000001""",1997-03-01,0.049014,5.038257,null,null,null
"""000001""",1997-04-01,0.087915,5.065245,null,null,null
"""000001""",1997-05-01,0.173395,4.976721,null,null,null
"""000001""",1997-06-01,-0.07327,5.067013,null,null,null
"""000001""",1997-07-01,-0.023041,5.181228,null,null,null
"""000001""",1997-08-01,-0.065068,5.168368,null,null,null
"""000001""",1997-09-01,-0.063837,5.421147,-0.572963,null,0.533567


连接数据 + 去重

In [97]:
ma_df = ma_df.join(fm_reg_controls_monthly_df, on=['date', 'portfolio'], how='left')
ma_df = ma_df.unique(['date','portfolio'])
ma_df.head()

date,portfolio,MA,return,betavals,ln_market_value,bm_ratio,gross_margin,investment_ratio
date,str,f64,f64,f64,f64,f64,f64,f64
2022-06-01,"""603909""",0.216544,-0.02742,-0.736995,-0.485759,-1.003291,0.006684,0.634474
2023-08-01,"""002922""",0.091194,0.005008,0.372355,-0.253505,null,null,null
2022-10-01,"""300240""",0.546748,-0.003436,0.410948,-0.505896,null,null,null
2024-10-01,"""301261""",1.790792,0.1124,0.149762,-1.138829,null,null,null
2017-05-01,"""000859""",0.66739,0.066,0.655978,-0.357807,null,null,null


缩尾


In [98]:
lo = pl.col("return").quantile(WINSORIZE_RATIO).over("date")
hi = pl.col("return").quantile(1 - WINSORIZE_RATIO).over("date")

ma_df = ma_df.with_columns(
    pl.col("return")
      .clip(
          pl.col("return").quantile(WINSORIZE_RATIO).over("date"),
          pl.col("return").quantile(1 - WINSORIZE_RATIO).over("date"),
      )
      .alias("return")
)

ma_df.head()

date,portfolio,MA,return,betavals,ln_market_value,bm_ratio,gross_margin,investment_ratio
date,str,f64,f64,f64,f64,f64,f64,f64
2022-06-01,"""603909""",0.216544,-0.02742,-0.736995,-0.485759,-1.003291,0.006684,0.634474
2023-08-01,"""002922""",0.091194,0.005008,0.372355,-0.253505,null,null,null
2022-10-01,"""300240""",0.546748,-0.003436,0.410948,-0.505896,null,null,null
2024-10-01,"""301261""",1.790792,0.1124,0.149762,-1.138829,null,null,null
2017-05-01,"""000859""",0.66739,0.066,0.655978,-0.357807,null,null,null


## 回归

定义一个FM回归的函数

In [99]:
def fm_reg(
    df:pl.DataFrame, 
    return_col:str, 
    x_cols:list[str],
    date_col:str='date',
    stkcd_col:str='portfolio'
) -> pl.DataFrame:
    """
    进行FM回归

    FM回归：
        - 1. 对于每一个时间的截面数据，进行线性回归
        - 2. 对于所有回归系数序列，求均值，HAC-t，p

    参数：
        df: 包含时间序列和截面数据的DataFrame
        return_col: 被解释变量列名
        x_cols: 解释变量列名列表
        date_col: 日期列名，默认为'date'
        stkcd_col: 股票代码列名，默认为'portfolio'（目前未使用，预留扩展）
    """
    # 确保返回列和自变量列存在
    if return_col not in df.columns:
        raise ValueError(f"返回列 '{return_col}' 不存在")

    for col in x_cols:
        if col not in df.columns:
            raise ValueError(f"自变量列 '{col}' 不存在")

    df = df.__copy__()
    df = df.drop_nulls()

    # 获取所有时间点
    time_points = df.select(date_col).unique().sort(date_col).to_series().to_list()

    # 存储每个时间点的回归系数
    coefficients_data = []

    for time_point in time_points:
        # 获取该时间点的截面数据
        cross_section = df.filter(pl.col(date_col) == time_point)

        # 跳过数据不足的时间点
        if len(cross_section) < len(x_cols) + 1:  # 需要至少比变量数多1个观测值
            warnings.warn(f"时间点 {time_point} 数据不足，跳过回归")
            continue

        # 准备回归数据
        y = cross_section.select(return_col).to_pandas().values.flatten()
        X = cross_section.select(x_cols).to_pandas().values

        # 添加常数项
        X = sm.add_constant(X)

        # 检查是否有足够的有效观测值
        valid_mask = ~np.isnan(y)
        for i in range(X.shape[1]):
            valid_mask = valid_mask & ~np.isnan(X[:, i])

        if np.sum(valid_mask) < len(x_cols) + 1:
            continue

        y_clean = y[valid_mask]
        X_clean = X[valid_mask]

        try:
            # 进行OLS回归
            model = OLS(y_clean, X_clean)
            results = model.fit()

            # 存储回归系数
            coeff_dict = {'date': time_point, 'const': results.params[0], 'n_obs': len(y_clean)}

            # 添加自变量系数
            for i, col in enumerate(x_cols):
                coeff_dict[col] = results.params[i + 1]

            coefficients_data.append(coeff_dict)

        except Exception as e:
            # 如果回归失败，跳过该时间点
            continue

    if not coefficients_data:
        raise ValueError("没有足够的有效数据进行回归")

    # 转换为DataFrame
    coeff_df = pd.DataFrame(coefficients_data)

    # 计算统计量
    stats_results = []

    # 对每个系数（包括常数项和自变量）计算统计量
    coeff_cols = ['const'] + x_cols

    for col in coeff_cols:
        if col not in coeff_df.columns:
            continue

        # 获取系数时间序列
        series = coeff_df[col].dropna()

        if len(series) < 2:
            continue

        # 计算HAC标准误差和t统计量
        try:
            y = series.to_numpy()
            x = np.ones((len(y), 1))
            results = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
            mean_coeff = results.params[0]
            t_stat = results.tvalues[0]
            p_value = results.pvalues[0]

        except Exception:
            warnings.warn(f"时间点 {time_point} 回归失败，跳过")
            t_stat = np.nan
            p_value = np.nan

        stats_results.append({
            'variable': col,
            'mean': mean_coeff,
            't_stat': t_stat,
            'p_value': p_value,
        })

        n_periods = len(series)

    # 转换为Polars DataFrame
    result_df = pl.DataFrame(stats_results)
    result_df = result_df.with_columns(
        pl.col('mean').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('mean'),
        pl.col('t_stat').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t_stat'),
        pl.col('p_value').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p_value'),
    )

    # t、p 加括号
    result_df = result_df.select(
        pl.col('variable'),
        pl.col('mean'),
        (pl.lit('[') + pl.col('t_stat') + pl.lit(']')).alias('t_stat'),
        (pl.lit('(') + pl.col('p_value') + pl.lit(')')).alias('p_value'),
    )
    # 居中对齐到 12 位
    result_df = result_df.with_columns(
        pl.col('mean').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('mean'),
        pl.col('t_stat').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t_stat'),
        pl.col('p_value').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p_value'),
    )

    # 合并为一行展示
    result_df = result_df.select(
        pl.col('variable'),
        (pl.col('mean') + pl.lit('\n') + pl.col('t_stat') + pl.lit('\n') + pl.col('p_value')).alias('stats'),
    )

    n_periods_df = pl.DataFrame(
        {
            'variable': ['观测数'],
            'stats': [n_periods]
        }
    ).with_columns(
        pl.col('stats').cast(pl.Utf8).alias('stats')
    )


    result_df = result_df.vstack(n_periods_df)


    return result_df

### 单AED变量

In [100]:
ma_result = fm_reg(ma_df, 'return', ['MA'])
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ma_result)

variable,stats
str,str
"""const""",""" 0.0005588 [0.05288] (0.9578) """
"""MA""",""" 0.01049 [8.404] (4.321e-17) """
"""观测数""","""81"""


### AED,BETA,SIZE,BM

In [101]:
ma_beta_size_bm_result = fm_reg(ma_df, 'return', ['MA','betavals','ln_market_value','bm_ratio'])
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ma_beta_size_bm_result)


variable,stats
str,str
"""const""",""" -0.0005248 [-0.0486] (0.9612) """
"""MA""",""" 0.0118 [11.03] (2.691e-28) """
"""betavals""",""" 0.01258 [1.765] (0.07764) """
"""ln_market_value""",""" 0.005899 [3.496] (0.0004731) """
"""bm_ratio""",""" 0.004007 [2.828] (0.004683) """
"""观测数""","""81"""


### AED,BETA,SIZE,BM,GM,IA


In [102]:
ma_beta_size_bm_gm_ia_result = fm_reg(ma_df, 'return', ['MA','betavals','ln_market_value','bm_ratio','gross_margin','investment_ratio'])
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ma_beta_size_bm_gm_ia_result)

variable,stats
str,str
"""const""",""" 0.0008734 [0.07421] (0.9408) """
"""MA""",""" 0.01173 [11.1] (1.286e-28) """
"""betavals""",""" 0.01389 [1.822] (0.06848) """
"""ln_market_value""",""" 0.005706 [3.393] (0.0006925) """
"""bm_ratio""",""" 0.004352 [3.149] (0.001641) """
"""gross_margin""",""" -0.0519 [-1.089] (0.2762) """
"""investment_ratio""",""" -0.0005395 [-0.7029] (0.4821) """
"""观测数""","""81"""


拼在一起

In [103]:
fm_result = ma_beta_size_bm_gm_ia_result.join(ma_beta_size_bm_result,on='variable',how='left')
fm_result = fm_result.rename({'stats':'5控制变量','stats_right':'3控制变量'})
fm_result = fm_result.join(ma_result,on='variable',how='left')
fm_result = fm_result.rename({'stats':'单控制变量'})
fm_result = fm_result.select(
    pl.col('variable'),
    pl.col('单控制变量'),
    pl.col('3控制变量'),
    pl.col('5控制变量'),
)

# 移动const
fm_result = fm_result[[1,2,3,4,5,6,0,7],:]


In [104]:
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(fm_result)

variable,单控制变量,3控制变量,5控制变量
str,str,str,str
"""MA""",""" 0.01049 [8.404] (4.321e-17) """,""" 0.0118 [11.03] (2.691e-28) """,""" 0.01173 [11.1] (1.286e-28) """
"""betavals""",null,""" 0.01258 [1.765] (0.07764) """,""" 0.01389 [1.822] (0.06848) """
"""ln_market_value""",null,""" 0.005899 [3.496] (0.0004731) """,""" 0.005706 [3.393] (0.0006925) """
"""bm_ratio""",null,""" 0.004007 [2.828] (0.004683) """,""" 0.004352 [3.149] (0.001641) """
"""gross_margin""",null,null,""" -0.0519 [-1.089] (0.2762) """
"""investment_ratio""",null,null,""" -0.0005395 [-0.7029] (0.4821) """
"""const""",""" 0.0005588 [0.05288] (0.9578) """,""" -0.0005248 [-0.0486] (0.9612) """,""" 0.0008734 [0.07421] (0.9408) """
"""观测数""","""81""","""81""","""81"""


In [105]:
if SAVE:
    fm_result.write_parquet(SAVE_BASE_DIR + '/FM回归.parquet')